# 🔍 Notebook 02 — Log Parsing

**Multimodal RCA Engine — Phase 1: Log Extraction & Processing**

This notebook covers:
1. Regex-based structured parsing for HDFS, BGL, and OpenStack logs
2. Log template mining using the **Drain** algorithm
3. Template analysis and EventId assignment
4. Export of parsed logs to CSV

---

## 2.1 — Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

from src.utils import (
    RAW_DIR, PARSED_DIR, FIGURES_DIR,
    ensure_directories, read_log_file, print_section, get_file_size_mb
)
from src.log_parser import RegexLogParser, DrainLogParser, extract_block_ids

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

ensure_directories()
print("✅ Setup complete!")

## 2.2 — Regex Parsing: HDFS Logs

HDFS log format: `<Date> <Time> <Pid> <Level> <Component>: <Content>`

Example:
```
081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
```

In [ ]:
# Find the main HDFS log file
hdfs_dir = RAW_DIR / "HDFS_v1"
hdfs_log_files = list(hdfs_dir.rglob('*.log'))
if not hdfs_log_files:
    hdfs_log_files = [f for f in hdfs_dir.rglob('*') if f.is_file() and f.stat().st_size > 10000
                      and f.suffix not in ['.csv', '.zip', '.gz', '.md', '.npz']]

print("HDFS log files found:")
for f in hdfs_log_files:
    print(f"  📄 {f.name} ({get_file_size_mb(f):.1f} MB)")

In [ ]:
# Parse HDFS logs with regex
# Using a sample for speed — adjust max_lines for full dataset
HDFS_SAMPLE_SIZE = 100_000  # Set to None for full dataset

hdfs_parser = RegexLogParser(log_format='hdfs')

if hdfs_log_files:
    hdfs_df = hdfs_parser.parse_file(hdfs_log_files[0], max_lines=HDFS_SAMPLE_SIZE)
    print(f"\n📊 Parsed DataFrame shape: {hdfs_df.shape}")
    display(hdfs_df.head(10))
else:
    print("❌ No HDFS log file found. Run Notebook 01 first.")

In [ ]:
# === HDFS Log Level Distribution ===
if 'hdfs_df' in dir() and not hdfs_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Level distribution
    level_counts = hdfs_df['level'].value_counts()
    colors_map = {'INFO': '#2196F3', 'WARN': '#FF9800', 'ERROR': '#F44336', 'FATAL': '#9C27B0', 'DEBUG': '#4CAF50'}
    bar_colors = [colors_map.get(l, '#9E9E9E') for l in level_counts.index]
    
    level_counts.plot(kind='bar', ax=ax1, color=bar_colors, edgecolor='white', linewidth=1.5)
    ax1.set_title('HDFS — Log Level Distribution', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Count')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
    
    # Top components
    top_components = hdfs_df['component'].value_counts().head(10)
    top_components.plot(kind='barh', ax=ax2, color='#1976D2', edgecolor='white')
    ax2.set_title('HDFS — Top 10 Components', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Count')
    ax2.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / '02_hdfs_parsed_stats.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 2.3 — Extract Block IDs (HDFS Sessions)

HDFS logs contain block IDs (`blk_xxxxx`) that identify individual data block operations. 
We use these as **session identifiers** for anomaly detection.

In [ ]:
if 'hdfs_df' in dir() and not hdfs_df.empty:
    # Extract block IDs
    hdfs_df['block_id'] = extract_block_ids(hdfs_df['content'])
    
    n_with_block = hdfs_df['block_id'].notna().sum()
    n_unique_blocks = hdfs_df['block_id'].nunique()
    
    print(f"📊 Block ID Extraction:")
    print(f"   Lines with block_id:    {n_with_block:,} / {len(hdfs_df):,} ({n_with_block/len(hdfs_df)*100:.1f}%)")
    print(f"   Unique block IDs:       {n_unique_blocks:,}")
    
    # Show sample
    display(hdfs_df[hdfs_df['block_id'].notna()][['content', 'block_id']].head(10))

## 2.4 — Regex Parsing: BGL Logs

In [ ]:
BGL_SAMPLE_SIZE = 50_000

bgl_dir = RAW_DIR / "BGL"
bgl_log_files = [f for f in bgl_dir.rglob('*') if f.is_file() and f.stat().st_size > 10000
                 and f.suffix not in ['.csv', '.zip', '.gz', '.md', '.npz']]

if bgl_log_files:
    print(f"📄 BGL main file: {bgl_log_files[0].name}")
    bgl_parser = RegexLogParser(log_format='bgl')
    bgl_df = bgl_parser.parse_file(bgl_log_files[0], max_lines=BGL_SAMPLE_SIZE)
    print(f"\n📊 Parsed DataFrame shape: {bgl_df.shape}")
    display(bgl_df.head(10))
else:
    print("❌ No BGL log file found. Run Notebook 01 first.")

In [ ]:
# === BGL Label Analysis ===
if 'bgl_df' in dir() and not bgl_df.empty and 'label' in bgl_df.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Alert vs non-alert
    bgl_df['is_alert'] = bgl_df['label'] != '-'
    alert_counts = bgl_df['is_alert'].value_counts()
    alert_counts.index = ['Non-Alert', 'Alert']
    
    alert_counts.plot(kind='bar', ax=ax1, color=['#4CAF50', '#F44336'], edgecolor='white')
    ax1.set_title('BGL — Alert vs Non-Alert', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Count')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
    
    # Log levels
    if 'level' in bgl_df.columns:
        bgl_df['level'].value_counts().head(10).plot(
            kind='barh', ax=ax2, color='#1976D2', edgecolor='white'
        )
        ax2.set_title('BGL — Log Severity Levels', fontsize=14, fontweight='bold')
        ax2.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / '02_bgl_parsed_stats.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 2.5 — Regex Parsing: OpenStack Logs

In [ ]:
OPENSTACK_SAMPLE_SIZE = 50_000

openstack_dir = RAW_DIR / "OpenStack"
openstack_log_files = [f for f in openstack_dir.rglob('*.log') if f.stat().st_size > 1000]
if not openstack_log_files:
    openstack_log_files = [f for f in openstack_dir.rglob('*') if f.is_file() 
                           and f.stat().st_size > 1000
                           and f.suffix not in ['.csv', '.zip', '.gz', '.md', '.tar']]

if openstack_log_files:
    print(f"📄 OpenStack log files ({len(openstack_log_files)}):")
    for f in openstack_log_files[:5]:
        print(f"  📄 {f.name} ({get_file_size_mb(f):.2f} MB)")
    
    openstack_parser = RegexLogParser(log_format='openstack')
    openstack_df = openstack_parser.parse_file(openstack_log_files[0], max_lines=OPENSTACK_SAMPLE_SIZE)
    print(f"\n📊 Parsed DataFrame shape: {openstack_df.shape}")
    display(openstack_df.head(10))
else:
    print("❌ No OpenStack log file found. Run Notebook 01 first.")

## 2.6 — Log Template Mining with Drain Algorithm

The **Drain** algorithm automatically discovers log templates (patterns) by building a parse tree.
Each unique template gets a `cluster_id` which becomes our **EventId**.

Reference: *He et al., "Drain: An Online Log Parsing Approach with Fixed Depth Tree", ICWS 2017*

In [ ]:
# === Drain Parsing on HDFS Content ===
if 'hdfs_df' in dir() and not hdfs_df.empty:
    print_section("Drain Template Mining — HDFS")
    
    drain_parser = DrainLogParser(depth=4, sim_th=0.4)
    
    # Process log content messages
    log_contents = hdfs_df['content'].dropna().tolist()
    drain_results = drain_parser.process_logs(log_contents)
    
    # Show summary
    drain_parser.get_summary()
    
    # Get templates DataFrame
    templates_df = drain_parser.get_templates_df()
    print(f"\n📋 Discovered Templates:")
    display(templates_df)

In [ ]:
# === Assign EventId to each log entry ===
if 'drain_results' in dir():
    # Add cluster_id as event_id to the dataframe
    event_ids = [r['cluster_id'] for r in drain_results]
    
    # We may have fewer drain results than rows (due to dropna)
    hdfs_parsed = hdfs_df[hdfs_df['content'].notna()].copy()
    hdfs_parsed['event_id'] = event_ids
    
    print(f"📊 HDFS with EventIds: {hdfs_parsed.shape}")
    display(hdfs_parsed[['level', 'component', 'content', 'block_id', 'event_id']].head(15))

In [ ]:
# === Template Distribution Visualization ===
if 'templates_df' in dir() and not templates_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Top 15 templates by frequency
    top_templates = templates_df.head(15)
    
    # Truncate template text for display
    display_names = [t[:60] + '...' if len(t) > 60 else t for t in top_templates['template']]
    
    bars = ax1.barh(range(len(top_templates)), top_templates['count'], color='#1976D2', edgecolor='white')
    ax1.set_yticks(range(len(top_templates)))
    ax1.set_yticklabels(display_names, fontsize=9)
    ax1.set_xlabel('Count')
    ax1.set_title('Top 15 Log Templates (Drain)', fontsize=14, fontweight='bold')
    ax1.invert_yaxis()
    
    # Template frequency distribution (log scale)
    ax2.hist(templates_df['count'], bins=30, color='#388E3C', edgecolor='white', alpha=0.8)
    ax2.set_xlabel('Template Frequency')
    ax2.set_ylabel('Number of Templates')
    ax2.set_title('Template Frequency Distribution', fontsize=14, fontweight='bold')
    ax2.set_yscale('log')
    
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / '02_drain_templates.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 2.7 — Export Parsed Data

In [ ]:
# Save parsed HDFS logs
if 'hdfs_parsed' in dir():
    output_path = PARSED_DIR / 'hdfs_parsed.csv'
    hdfs_parsed.to_csv(output_path, index=False)
    print(f"💾 Saved: {output_path} ({get_file_size_mb(output_path):.1f} MB)")

# Save templates
if 'templates_df' in dir():
    templates_path = PARSED_DIR / 'hdfs_templates.csv'
    templates_df.to_csv(templates_path, index=False)
    print(f"💾 Saved: {templates_path}")

# Save BGL if parsed
if 'bgl_df' in dir() and not bgl_df.empty:
    bgl_output = PARSED_DIR / 'bgl_parsed.csv'
    bgl_df.to_csv(bgl_output, index=False)
    print(f"💾 Saved: {bgl_output}")

# Save OpenStack if parsed
if 'openstack_df' in dir() and not openstack_df.empty:
    os_output = PARSED_DIR / 'openstack_parsed.csv'
    openstack_df.to_csv(os_output, index=False)
    print(f"💾 Saved: {os_output}")

print("\n✅ All parsed data exported successfully!")

## 2.8 — Summary & Next Steps

### ✅ What we accomplished:
- Parsed HDFS, BGL, and OpenStack logs with regex into structured DataFrames
- Extracted HDFS block IDs for session grouping
- Discovered log templates using the Drain algorithm
- Assigned EventIds to each log entry
- Exported all parsed data to CSV

### 📊 Key Metrics:
| Metric | Value |
|--------|-------|
| HDFS templates discovered | See Drain output above |
| Regex success rate | See parsing stats above |
| Unique block IDs | See block extraction above |

### ➡️ Next: Notebook 03 — Feature Engineering
We'll build feature vectors from the parsed logs and align them with anomaly labels.